# Check Point 2 — Classificação binária de imagens com CNN: Gatos × Cachorros

**Disciplina:** Aprendizado Profundo Convolucional com Redes Neurais Convolucionais — FIAP

**Tema:** preparação, fluxo interno e interpretação de uma CNN simples

## 1. Identificação

| Integrante (nome completo) | RM |
|---|---|
| PREENCHER NOME COMPLETO | RM PREENCHER |
| PREENCHER NOME COMPLETO | RM PREENCHER |
| PREENCHER NOME COMPLETO | RM PREENCHER |

**Turma:** PREENCHER

**Data de entrega:** PREENCHER

## 2. Problema e base de imagens

**Problema:** dada uma fotografia, decidir se ela mostra um **gato** ou um **cachorro** (classificação binária).

**Fonte dos dados:** *Dog and Cat Classification Dataset*, publicado no Kaggle por Bhavik Jikadara
(<https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset>). É uma redistribuição da base
*Cats vs Dogs* da Microsoft/Petfinder (Asirra), baixada aqui com a biblioteca `kagglehub`.

**Por que essa base?**

- possui **exatamente duas classes** (`Cat` e `Dog`), já rotuladas pelo nome da pasta — não exige bounding boxes nem máscaras;
- tem origem identificável e é pública;
- é balanceada e tem imagens de sobra para separar treino, validação e teste;
- é um problema intuitivo para humanos, mas com bastante variação de pose, fundo, iluminação e raça, o que torna visível
  o que uma CNN simples consegue (e não consegue) aprender **sem** transfer learning.

**Classes:**

| Rótulo | Pasta | O que representa |
|---|---|---|
| 0 | `Cat` | fotos de gatos domésticos (classe negativa) |
| 1 | `Dog` | fotos de cachorros domésticos (classe positiva — a saída sigmoid estima `P(cachorro)`) |

In [ ]:
import os
import random

os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import Markdown, display

import keras
from keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

print("Keras:", keras.__version__, "| backend:", keras.backend.backend())

In [ ]:
# Instale com `pip install kagglehub` caso o ambiente ainda não tenha a biblioteca (o Google Colab já tem).
import kagglehub

path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")
print("Path to dataset files:", path)

In [ ]:
CLASSES = ["Cat", "Dog"]                     # índice na lista = rótulo numérico
NOMES_PT = {0: "Gato", 1: "Cachorro"}
EXTENSOES = (".jpg", ".jpeg", ".png")

# Localiza a pasta que contém as subpastas Cat/ e Dog/
DATA_DIR = next(raiz for raiz, pastas, _ in os.walk(path) if all(c in pastas for c in CLASSES))
print("Pasta das classes:", DATA_DIR)

arquivos = {
    rotulo: sorted(os.path.join(DATA_DIR, classe, f)
                   for f in os.listdir(os.path.join(DATA_DIR, classe))
                   if f.lower().endswith(EXTENSOES))
    for rotulo, classe in enumerate(CLASSES)
}

contagem = pd.DataFrame({
    "rótulo": list(arquivos.keys()),
    "classe": [f"{CLASSES[r]} ({NOMES_PT[r]})" for r in arquivos],
    "imagens": [len(v) for v in arquivos.values()],
})
contagem["proporção (%)"] = (100 * contagem["imagens"] / contagem["imagens"].sum()).round(2)
print("Total de imagens na base:", contagem["imagens"].sum())
contagem

### Subconjunto utilizado

A base completa é grande para ser carregada inteira em memória como tensores `float32`. Como o objetivo do trabalho é
entender o fluxo de **uma** CNN simples (e não maximizar acurácia), sorteamos um **subconjunto balanceado** com a mesma
quantidade de imagens por classe. O sorteio usa semente fixa, então o notebook é reprodutível.
Arquivos corrompidos ou ilegíveis, se existirem, são descartados nessa etapa.

In [ ]:
N_POR_CLASSE = 4000


def imagem_valida(caminho):
    """Retorna True se o arquivo pode ser aberto como imagem."""
    try:
        with Image.open(caminho) as img:
            img.verify()
        return True
    except Exception:
        return False


rng = random.Random(SEED)
caminhos, rotulos = [], []
for rotulo, lista in arquivos.items():
    candidatos = lista.copy()
    rng.shuffle(candidatos)
    escolhidos = []
    for c in candidatos:
        if imagem_valida(c):
            escolhidos.append(c)
        if len(escolhidos) == N_POR_CLASSE:
            break
    caminhos += escolhidos
    rotulos += [rotulo] * len(escolhidos)

caminhos = np.array(caminhos)
rotulos = np.array(rotulos)

print("Imagens utilizadas no projeto:", len(caminhos))
for r in NOMES_PT:
    print(f"  classe {r} - {NOMES_PT[r]:8s}: {(rotulos == r).sum()} imagens")

plt.figure(figsize=(4, 3))
plt.bar([NOMES_PT[r] for r in NOMES_PT], [(rotulos == r).sum() for r in NOMES_PT], color=["#4C72B0", "#DD8452"])
plt.title("Distribuição das classes (subconjunto utilizado)")
plt.ylabel("nº de imagens")
plt.show()

In [ ]:
# Exemplos visuais: 4 imagens de cada classe, ainda no tamanho original
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for linha, rotulo in enumerate(NOMES_PT):
    exemplos = caminhos[rotulos == rotulo][:4]
    for ax, caminho in zip(axes[linha], exemplos):
        img = Image.open(caminho).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{NOMES_PT[rotulo]} — {img.size[1]}×{img.size[0]} px", fontsize=10)
        ax.axis("off")
plt.suptitle("Exemplos de cada classe (tamanho original, altura × largura)", fontsize=13)
plt.tight_layout()
plt.show()

Já dá para notar que as imagens têm **tamanhos e proporções diferentes**, fundos variados e animais em poses e
distâncias distintas. É por isso que a etapa de preparação (Seção 3) é necessária antes de qualquer treinamento.

## 3. Inspeção e preparação dos dados

### 3.1 A imagem como tensor H × W × C

In [ ]:
caminho_exemplo = caminhos[rotulos == 1][0]
img_original = np.array(Image.open(caminho_exemplo).convert("RGB"))

H, W, C = img_original.shape
print("Arquivo :", os.path.basename(caminho_exemplo), "| classe:", NOMES_PT[1])
print("shape   :", img_original.shape)
print("dtype   :", img_original.dtype)
print("mínimo  :", img_original.min())
print("máximo  :", img_original.max())
print()
print(f"Altura  (H) = {H} -> a imagem tem {H} linhas de pixels")
print(f"Largura (W) = {W} -> cada linha tem {W} pixels (colunas)")
print(f"Canais  (C) = {C} -> cada pixel guarda {C} valores: R, G e B")
print(f"Total de números no tensor: {H} x {W} x {C} = {img_original.size}")
print()
print("Pixel da posição (linha 0, coluna 0) -> [R G B] =", img_original[0, 0])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img_original)
axes[0].set_title(f"RGB — shape {img_original.shape}")
for i, (nome, cmap) in enumerate(zip(["R (vermelho)", "G (verde)", "B (azul)"], ["Reds", "Greens", "Blues"])):
    axes[i + 1].imshow(img_original[:, :, i], cmap=cmap, vmin=0, vmax=255)
    axes[i + 1].set_title(f"Canal {nome} — shape {img_original[:, :, i].shape}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

**Como ler o tensor H × W × C (com base no exemplo acima):**

- **H (altura)** — primeiro eixo: quantidade de **linhas** de pixels, de cima para baixo.
- **W (largura)** — segundo eixo: quantidade de **colunas** de pixels, da esquerda para a direita.
- **C (canais)** — terceiro eixo: quantos valores descrevem cada pixel. Numa imagem RGB são 3: intensidade de
  vermelho, verde e azul. Cada canal sozinho é uma matriz H × W (as três figuras coloridas acima).

O `dtype` é `uint8`: cada valor é um inteiro de **0 a 255** (0 = ausência daquela cor, 255 = intensidade máxima).
Ou seja, para o computador a foto é apenas um bloco tridimensional de números — e é esse bloco que a CNN recebe.

### 3.2 Divisão em treino, validação e teste

A divisão é feita **antes** de qualquer treinamento, sobre os caminhos dos arquivos, de forma **estratificada**
(mantém 50 % / 50 % em cada conjunto):

- **70 % treino** — usado para ajustar os pesos;
- **15 % validação** — usado para acompanhar as curvas durante o treino;
- **15 % teste** — guardado e usado **somente** na avaliação final (Seção 7).

In [ ]:
cam_treino, cam_resto, y_treino, y_resto = train_test_split(
    caminhos, rotulos, test_size=0.30, stratify=rotulos, random_state=SEED)
cam_val, cam_teste, y_val, y_teste = train_test_split(
    cam_resto, y_resto, test_size=0.50, stratify=y_resto, random_state=SEED)

divisao = pd.DataFrame({
    "conjunto": ["treino", "validação", "teste"],
    "total": [len(y_treino), len(y_val), len(y_teste)],
    "gatos (0)": [(y_treino == 0).sum(), (y_val == 0).sum(), (y_teste == 0).sum()],
    "cachorros (1)": [(y_treino == 1).sum(), (y_val == 1).sum(), (y_teste == 1).sum()],
})
divisao

### 3.3 Redimensionamento e normalização

Uma **única função** de pré-processamento é aplicada aos três conjuntos, garantindo exatamente o mesmo tratamento:

1. converte para RGB (algumas fotos da base vêm em tons de cinza ou CMYK) → sempre 3 canais;
2. redimensiona para **128 × 128** pixels;
3. converte para `float32` e divide por 255 → pixels na faixa **[0, 1]**.

In [ ]:
IMG_SIZE = 128


def preparar_imagem(caminho, tamanho=IMG_SIZE):
    """Pré-processamento único, usado em treino, validação e teste."""
    img = Image.open(caminho).convert("RGB")                    # sempre 3 canais
    img = img.resize((tamanho, tamanho), Image.BILINEAR)        # tamanho fixo
    return np.asarray(img, dtype=np.float32) / 255.0            # normalização para [0, 1]


def carregar_conjunto(lista_caminhos):
    return np.stack([preparar_imagem(c) for c in lista_caminhos])


X_treino = carregar_conjunto(cam_treino)
X_val = carregar_conjunto(cam_val)
X_teste = carregar_conjunto(cam_teste)

y_treino = y_treino.astype("float32")
y_val = y_val.astype("float32")
y_teste = y_teste.astype("float32")

for nome, X, y in [("treino", X_treino, y_treino), ("validação", X_val, y_val), ("teste", X_teste, y_teste)]:
    print(f"{nome:10s} X: {X.shape} {X.dtype} | min={X.min():.2f} max={X.max():.2f} | y: {y.shape}")

In [ ]:
# Antes e depois do pré-processamento, para a mesma imagem inspecionada na Seção 3.1
img_preparada = preparar_imagem(caminho_exemplo)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(img_original)
axes[0].set_title(f"ANTES\nshape={img_original.shape} | dtype={img_original.dtype}\n"
                  f"min={img_original.min()} | max={img_original.max()}")
axes[1].imshow(img_preparada)
axes[1].set_title(f"DEPOIS (resize + normalização)\nshape={img_preparada.shape} | dtype={img_preparada.dtype}\n"
                  f"min={img_preparada.min():.3f} | max={img_preparada.max():.3f}")
plt.tight_layout()
plt.show()

**Por que o modelo precisa de entrada padronizada?**

- **Tamanho fixo:** a rede recebe lotes (*batches*) empilhados num único tensor `(batch, 128, 128, 3)`; isso só é
  possível se todas as imagens tiverem o mesmo shape. Além disso, o tamanho de entrada determina o tamanho dos mapas
  de características em todas as camadas seguintes.
- **Normalização para [0, 1]:** valores entre 0 e 255 geram ativações e gradientes muito grandes logo no início;
  com entradas pequenas e na mesma escala o otimizador converge de forma mais rápida e estável.
- **Mesma regra nos três conjuntos:** se validação ou teste fossem preparados de outra forma, o modelo veria dados
  numa escala/tamanho diferente da que aprendeu, e as métricas deixariam de ser confiáveis.

O conteúdo visual é preservado (a imagem "depois" é a mesma cena, só com menos resolução e levemente deformada pela
mudança de proporção); o que muda é a **representação numérica**.

> **Data augmentation (opcional):** usamos um aumento leve — espelhamento horizontal —
> implementado como uma camada *dentro* do modelo (Seção 5). Esse tipo de camada do Keras só atua durante o `fit`
> (`training=True`); na validação, no teste e no `predict` ela apenas repassa a imagem. Logo, o aumento é
> aplicado **somente ao treino**. Um gato ou cachorro espelhado continua sendo uma imagem plausível.

## 4. Convolução demonstrativa (filtro 3 × 3 manual)

Antes de treinar a CNN, aplicamos **manualmente** um filtro 3 × 3 a uma imagem da base para ver, na prática, o que
uma convolução faz. O kernel escolhido é o **Sobel horizontal (Gx)**, um detector de **bordas verticais**:

$$K = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}$$

Ele subtrai os vizinhos da esquerda dos vizinhos da direita. Em regiões uniformes a soma dá ≈ 0; onde a intensidade
muda bruscamente da esquerda para a direita (uma borda vertical), o resultado é grande em módulo.

A operação é feita em tons de cinza (1 canal), sem padding e com stride 1 — por isso a saída perde 1 pixel em cada lado.

In [ ]:
kernel = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]], dtype=np.float32)


def convolucao2d(imagem, kernel):
    """Convolução 'valid' com stride 1, implementada explicitamente (como a Conv2D faz, sem inverter o kernel)."""
    kh, kw = kernel.shape
    h_out = imagem.shape[0] - kh + 1
    w_out = imagem.shape[1] - kw + 1
    saida = np.zeros((h_out, w_out), dtype=np.float32)
    for i in range(h_out):
        for j in range(w_out):
            janela = imagem[i:i + kh, j:j + kw]           # região local 3x3
            saida[i, j] = np.sum(janela * kernel)         # multiplica elemento a elemento e soma
    return saida


img_cinza = img_preparada.mean(axis=-1)                    # 128 x 128, valores em [0, 1]
mapa = convolucao2d(img_cinza, kernel)

print("Kernel utilizado (Sobel Gx):\n", kernel)
print("\nEntrada :", img_cinza.shape)
print("Saída   :", mapa.shape, "-> (128 - 3 + 1) = 126 em cada dimensão")
print(f"Valores do mapa: min={mapa.min():.2f} | max={mapa.max():.2f}")

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
axes[0].imshow(img_preparada)
axes[0].set_title("Imagem original (128×128×3)")
axes[1].imshow(img_cinza, cmap="gray")
axes[1].set_title("Entrada da convolução (cinza, 128×128)")
im = axes[2].imshow(mapa, cmap="seismic", vmin=-abs(mapa).max(), vmax=abs(mapa).max())
axes[2].set_title("Mapa resultante (126×126)\nazul = claro→escuro | vermelho = escuro→claro")
plt.colorbar(im, ax=axes[2], fraction=0.046)
axes[3].imshow(np.abs(mapa), cmap="gray")
axes[3].set_title("|mapa| — intensidade da borda")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

**O que mudou na imagem?**

- As regiões lisas (pelagem uniforme, fundo homogêneo) ficaram próximas de **zero** — o filtro "apagou" tudo o que não
  varia localmente.
- Os **contornos verticais** (laterais do corpo e da cabeça, orelhas, limites entre o animal e o fundo, objetos do
  cenário) ficaram com valores altos em módulo. O sinal indica o sentido da transição: positivo quando a imagem passa
  de escuro para claro (da esquerda para a direita) e negativo no sentido contrário.
- Bordas **horizontais** quase não aparecem, porque esse kernel só compara esquerda × direita.

**Relação com mapa de características:** a matriz 126 × 126 obtida é exatamente um *feature map*: cada posição
responde à pergunta "o padrão que este kernel procura está presente **aqui**?". Uma camada `Conv2D` faz a mesma operação,
com duas diferenças: (1) ela aplica **vários** kernels em paralelo, gerando um mapa por filtro (os canais da saída), e
(2) os valores dos kernels **não são definidos à mão** — são pesos aprendidos durante o treinamento.

## 5. Arquitetura da CNN

Uma **única** CNN, definida pelo grupo, com quatro blocos `Conv2D → BatchNormalization → MaxPooling2D`, seguidos de
`GlobalAveragePooling2D` e de uma pequena parte densa.

| Componente | Configuração | Papel na arquitetura |
|---|---|---|
| `Input` | `(128, 128, 3)` | Define o tensor de entrada, compatível com o resize e com os 3 canais RGB. |
| `RandomFlip` | horizontal | Data augmentation leve; **só atua no treino** e não altera o shape. |
| `Conv2D` (×4) | 32, 64, 128 e 128 filtros; `kernel_size=3`; `padding="same"`; `strides=1`; `activation="relu"` | Extraem padrões locais. Os primeiros filtros aprendem bordas e cores; os mais profundos combinam esses padrões em texturas e partes (orelhas, focinho, olhos). `same` preserva H e W; a ReLU zera respostas negativas e introduz não linearidade. |
| `BatchNormalization` (×4) | padrão | Padroniza a saída de cada convolução (média ≈ 0, desvio ≈ 1 por canal, calculados no batch). Estabiliza e acelera o treino; **não altera o shape**. |
| `MaxPooling2D` (×4) | `pool_size=2` (stride 2) | Reduz H e W pela metade mantendo a resposta mais forte de cada janela 2×2: diminui o custo, amplia o campo receptivo das camadas seguintes e dá pequena tolerância a deslocamentos. |
| `GlobalAveragePooling2D` | — | Resume cada mapa 8×8 em **um** valor (sua média), gerando um vetor de 128 posições. Escolhido no lugar do `Flatten` por ter muito menos parâmetros na camada densa seguinte (128 entradas em vez de 8·8·128 = 8192), reduzindo o risco de overfitting. |
| `Dropout` | 0.3 | Regularização: desliga aleatoriamente 30 % das entradas da parte densa durante o treino. |
| `Dense` | 64 neurônios, ReLU | Combina as 128 características resumidas. |
| `Dense` (saída) | 1 neurônio, **sigmoid** | Produz `P(cachorro)` entre 0 e 1. |

O número de filtros cresce (32 → 64 → 128) enquanto a resolução espacial diminui: a rede troca "onde" por "o quê".

### 5.1 Previsão dos shapes (feita **antes** do `model.summary()`)

Regras utilizadas:

- **Conv2D com `padding="same"` e `stride=1`:** H e W não mudam; o número de **canais passa a ser o número de filtros**.
  (Com `padding="valid"` seria `H_out = H − k + 1`, como na convolução manual: 128 → 126.)
- **MaxPooling2D 2×2 (stride 2):** `H_out = ⌊H / 2⌋`, `W_out = ⌊W / 2⌋`; o número de canais **não muda**.
- **GlobalAveragePooling2D:** `(H, W, C) → (C,)`.
- **Dense(n):** `(…) → (n,)`.
- **BatchNormalization e Dropout:** não alteram o shape.
- Parâmetros de uma Conv2D: `(k · k · canais_entrada + 1) · filtros`. Cada BatchNormalization guarda 4 valores por canal
  (escala e deslocamento, treináveis; média e variância móveis, não treináveis).

| Etapa | Conta | Shape esperado | Parâmetros esperados |
|---|---|---|---|
| Entrada | resize 128×128, RGB | (128, 128, 3) | 0 |
| Bloco 1 — Conv2D(32) | same → 128×128; 32 filtros | (128, 128, 32) | (3·3·3 + 1)·32 = 896 |
| Bloco 1 — MaxPool | 128 / 2 | (64, 64, 32) | 0 |
| Bloco 2 — Conv2D(64) | same → 64×64; 64 filtros | (64, 64, 64) | (3·3·32 + 1)·64 = 18 496 |
| Bloco 2 — MaxPool | 64 / 2 | (32, 32, 64) | 0 |
| Bloco 3 — Conv2D(128) | same → 32×32; 128 filtros | (32, 32, 128) | (3·3·64 + 1)·128 = 73 856 |
| Bloco 3 — MaxPool | 32 / 2 | (16, 16, 128) | 0 |
| Bloco 4 — Conv2D(128) | same → 16×16; 128 filtros | (16, 16, 128) | (3·3·128 + 1)·128 = 147 584 |
| Bloco 4 — MaxPool | 16 / 2 | (8, 8, 128) | 0 |
| GlobalAveragePooling2D | média de cada mapa 8×8 | (128,) | 0 |
| Dense(64) | 64 neurônios | (64,) | 128·64 + 64 = 8 256 |
| Saída — Dense(1, sigmoid) | 1 neurônio | (1,) | 64 + 1 = 65 |
| BatchNormalization (×4) | não mudam o shape | igual ao da Conv2D anterior | 4·(32 + 64 + 128 + 128) = 1 408 |
| **Total** | | | **250 561** (704 não treináveis) |

A célula abaixo registra essa previsão em código, para ser comparada automaticamente com o modelo construído.

In [ ]:
# Previsão feita à mão (sem consultar o modelo): nome da camada -> (shape esperado, interpretação)
previsao = {
    "entrada": ((128, 128, 3),   "Imagem RGB redimensionada e normalizada"),
    "conv1":   ((128, 128, 32),  "padding same mantém 128x128; 32 filtros -> 32 canais"),
    "pool1":   ((64, 64, 32),    "pooling 2x2 divide H e W por 2; canais iguais"),
    "conv2":   ((64, 64, 64),    "H e W mantidos; 64 filtros -> 64 canais"),
    "pool2":   ((32, 32, 64),    "H e W caem para 32"),
    "conv3":   ((32, 32, 128),   "H e W mantidos; 128 filtros -> 128 canais"),
    "pool3":   ((16, 16, 128),   "H e W caem para 16"),
    "conv4":   ((16, 16, 128),   "H e W mantidos; 128 filtros"),
    "pool4":   ((8, 8, 128),     "H e W caem para 8: 128 mapas de 8x8"),
    "gap":     ((128,),          "média de cada mapa 8x8 -> vetor com 1 valor por canal"),
    "dense":   ((64,),           "combina as 128 características em 64"),
    "saida":   ((1,),            "probabilidade da classe positiva (cachorro)"),
}

### 5.2 Construção do modelo

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="entrada"),

    # Data augmentation: ativo apenas durante o treino (model.fit)
    layers.RandomFlip("horizontal", seed=SEED, name="aug_flip"),

    # Bloco 1
    layers.Conv2D(32, kernel_size=3, strides=1, padding="same", activation="relu", name="conv1"),
    layers.BatchNormalization(name="bn1"),
    layers.MaxPooling2D(pool_size=2, name="pool1"),
    # Bloco 2
    layers.Conv2D(64, kernel_size=3, strides=1, padding="same", activation="relu", name="conv2"),
    layers.BatchNormalization(name="bn2"),
    layers.MaxPooling2D(pool_size=2, name="pool2"),
    # Bloco 3
    layers.Conv2D(128, kernel_size=3, strides=1, padding="same", activation="relu", name="conv3"),
    layers.BatchNormalization(name="bn3"),
    layers.MaxPooling2D(pool_size=2, name="pool3"),
    # Bloco 4
    layers.Conv2D(128, kernel_size=3, strides=1, padding="same", activation="relu", name="conv4"),
    layers.BatchNormalization(name="bn4"),
    layers.MaxPooling2D(pool_size=2, name="pool4"),

    # Mapas de características -> vetor
    layers.GlobalAveragePooling2D(name="gap"),
    layers.Dropout(0.3, name="dropout"),

    # Classificação
    layers.Dense(64, activation="relu", name="dense"),
    layers.Dense(1, activation="sigmoid", name="saida"),
], name="cnn_gatos_cachorros")

In [ ]:
model.summary()

### 5.3 Conferência: shape esperado × shape obtido

In [ ]:
obtido = {"entrada": tuple(model.input_shape[1:])}
for camada in model.layers:
    obtido[camada.name] = tuple(camada.output.shape[1:])       # remove a dimensão do batch (None)

rotulos_etapa = {
    "entrada": "Entrada",
    "conv1": "Bloco 1 — Conv2D(32)", "pool1": "Bloco 1 — MaxPooling2D",
    "conv2": "Bloco 2 — Conv2D(64)", "pool2": "Bloco 2 — MaxPooling2D",
    "conv3": "Bloco 3 — Conv2D(128)", "pool3": "Bloco 3 — MaxPooling2D",
    "conv4": "Bloco 4 — Conv2D(128)", "pool4": "Bloco 4 — MaxPooling2D",
    "gap": "Representação antes da saída — GAP",
    "dense": "Dense(64)", "saida": "Saída — Dense(1, sigmoid)",
}

tabela_shapes = pd.DataFrame([
    {
        "Etapa / camada": rotulos_etapa[nome],
        "Shape esperado": str(esperado),
        "Shape obtido": str(obtido[nome]),
        "Confere?": "sim" if esperado == obtido[nome] else "NÃO",
        "Interpretação": interpretacao,
    }
    for nome, (esperado, interpretacao) in previsao.items()
])

print("Parâmetros esperados: 250561 | obtidos:", model.count_params())
assert model.count_params() == 250561
assert (tabela_shapes["Confere?"] == "sim").all(), "Alguma previsão de shape não bateu!"
pd.set_option("display.max_colwidth", None)
tabela_shapes

Todas as previsões conferem com o modelo construído, inclusive o total de parâmetros. Lendo a tabela de cima para baixo:

- **altura e largura** só mudam nos `MaxPooling2D`: 128 → 64 → 32 → 16 → 8;
- o **número de canais** só muda nas `Conv2D` e é sempre igual ao número de filtros: 3 → 32 → 64 → 128 → 128;
- o `GlobalAveragePooling2D` elimina as duas dimensões espaciais e deixa apenas os 128 canais;
- a camada de augmentation, as `BatchNormalization` e o `Dropout` não alteram o shape (por isso não aparecem na tabela).

## 6. Treinamento

| Configuração | Valor | Justificativa |
|---|---|---|
| Função de perda | `binary_crossentropy` | Perda padrão para saída sigmoid com rótulos 0/1. |
| Otimizador | `Adam` (learning rate 1e-3) | Adapta o passo por parâmetro; funciona bem sem ajuste fino. |
| Métrica | acurácia | Classes balanceadas, então acurácia é informativa. |
| Épocas | 25 | Suficiente para observar a tendência das curvas. |
| Batch size | 32 | Equilíbrio entre estabilidade do gradiente e uso de memória. |

O modelo é treinado **uma única vez**, usando treino e validação. O conjunto de teste não é tocado aqui.

In [ ]:
EPOCAS = 25
BATCH_SIZE = 32

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

historico = model.fit(
    X_treino, y_treino,
    validation_data=(X_val, y_val),
    epochs=EPOCAS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    verbose=2,
)

In [ ]:
hist = pd.DataFrame(historico.history)
hist.index = hist.index + 1
hist.index.name = "época"

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(hist.index, hist["loss"], "o-", label="treino")
axes[0].plot(hist.index, hist["val_loss"], "o-", label="validação")
axes[0].set_title("Loss (binary crossentropy)")
axes[0].set_xlabel("época")
axes[0].set_ylabel("loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(hist.index, hist["accuracy"], "o-", label="treino")
axes[1].plot(hist.index, hist["val_accuracy"], "o-", label="validação")
axes[1].set_title("Acurácia")
axes[1].set_xlabel("época")
axes[1].set_ylabel("acurácia")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

melhor = hist["val_loss"].idxmin()
print(f"Épocas: {EPOCAS} | batch size: {BATCH_SIZE}")
print(f"Última época  -> treino: loss={hist['loss'].iloc[-1]:.4f} acc={hist['accuracy'].iloc[-1]:.4f} | "
      f"validação: loss={hist['val_loss'].iloc[-1]:.4f} acc={hist['val_accuracy'].iloc[-1]:.4f}")
print(f"Menor val_loss -> época {melhor}: val_loss={hist.loc[melhor, 'val_loss']:.4f} "
      f"val_acc={hist.loc[melhor, 'val_accuracy']:.4f}")
print(f"Diferença final (acc treino - acc validação): "
      f"{hist['accuracy'].iloc[-1] - hist['val_accuracy'].iloc[-1]:+.4f}")

### Diagnóstico das curvas

A célula abaixo lê o histórico **desta execução** e escreve o diagnóstico a partir de três sinais:

- **acurácia de treino baixa e próxima da validação** → *underfitting* (o modelo ainda não aprendeu o suficiente);
- **treino bem acima da validação**, ou **loss de validação voltando a subir** enquanto a de treino continua caindo → *overfitting*;
- **curvas próximas, com loss de validação caindo ou estável** → comportamento adequado.

In [ ]:
acc_tr, acc_val = hist["accuracy"].iloc[-1], hist["val_accuracy"].iloc[-1]
loss_tr, loss_val = hist["loss"].iloc[-1], hist["val_loss"].iloc[-1]
gap = acc_tr - acc_val
epoca_min = int(hist["val_loss"].idxmin())
val_loss_min = hist["val_loss"].min()
val_loss_subiu = (loss_val > 1.15 * val_loss_min) and (epoca_min <= EPOCAS - 4)

if acc_tr < 0.70 and gap < 0.05:
    tipo_curva = "underfitting"
    diagnostico_curvas = (
        f"As curvas indicam **underfitting**: mesmo no treino a acurácia final é baixa ({acc_tr:.1%}) e fica próxima "
        f"da validação ({acc_val:.1%}). O modelo ainda não extraiu padrões suficientes — as duas curvas de loss "
        f"continuam em queda, sinal de que mais épocas ou mais capacidade ainda trariam ganho.")
elif gap > 0.08 or val_loss_subiu:
    tipo_curva = "overfitting"
    diagnostico_curvas = (
        f"As curvas indicam **overfitting**. No treino a loss continua caindo e a acurácia chega a {acc_tr:.1%}, "
        f"mas na validação a acurácia fica em {acc_val:.1%} (diferença de {gap * 100:.1f} pontos percentuais). "
        f"A menor loss de validação ocorreu na época {epoca_min} ({val_loss_min:.3f}) e terminou em {loss_val:.3f}: "
        f"a partir desse ponto o modelo passou a memorizar particularidades do treino que não se repetem em imagens novas.")
else:
    tipo_curva = "adequado"
    diagnostico_curvas = (
        f"As curvas indicam **aprendizado adequado**: loss de treino e de validação caem juntas e terminam próximas "
        f"({loss_tr:.3f} × {loss_val:.3f}); a acurácia final é {acc_tr:.1%} no treino e {acc_val:.1%} na validação "
        f"(diferença de {gap * 100:.1f} pontos percentuais). Não há sinal forte de memorização do treino nem de "
        f"incapacidade de aprender.")

oscilacao = hist["val_loss"].diff().abs().mean()
observacao = (
    f"\n\nObservação: a curva de validação oscila de uma época para outra (variação média de {oscilacao:.3f} na loss). "
    f"Isso é esperado com `BatchNormalization` e um conjunto de validação pequeno ({len(y_val)} imagens): "
    f"a tendência geral importa mais do que o valor de uma época isolada.")

display(Markdown(diagnostico_curvas + observacao))

### 6.1 Olhando dentro da rede: mapas de características aprendidos

Na Seção 4 o kernel foi definido à mão. Agora os kernels foram **aprendidos**. Abaixo estão alguns mapas produzidos
pela primeira e pela última camada convolucional para a mesma imagem de exemplo.

In [ ]:
extrator = keras.Model(inputs=model.inputs,
                       outputs=[model.get_layer("conv1").output, model.get_layer("conv4").output])
mapas_conv1, mapas_conv4 = extrator.predict(img_preparada[None, ...], verbose=0)
print("Saída da conv1:", mapas_conv1.shape, "| Saída da conv4:", mapas_conv4.shape)

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i in range(8):
    axes[0, i].imshow(mapas_conv1[0, :, :, i], cmap="viridis")
    axes[0, i].set_title(f"conv1 — filtro {i}", fontsize=9)
    axes[1, i].imshow(mapas_conv4[0, :, :, i], cmap="viridis")
    axes[1, i].set_title(f"conv4 — filtro {i}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Mapas de características: conv1 (128×128, detalhes locais) × conv4 (16×16, padrões abstratos)")
plt.tight_layout()
plt.show()

Os mapas da `conv1` ainda lembram a imagem original: cada filtro realça bordas em certa orientação, regiões claras ou
escuras — muito parecido com o resultado do Sobel manual. Na `conv4` os mapas têm apenas 16 × 16 posições e já não são
reconhecíveis como foto: cada um indica **em que região** um padrão mais complexo foi encontrado. É essa informação
resumida que o GAP entrega à parte densa.

## 7. Avaliação no conjunto de teste

Somente agora o conjunto de teste é utilizado. A saída do modelo é `P(cachorro)`; a classe prevista é
**Cachorro (1)** se a probabilidade for ≥ 0,5 e **Gato (0)** caso contrário.

In [ ]:
prob_teste = model.predict(X_teste, batch_size=BATCH_SIZE, verbose=0).ravel()
pred_teste = (prob_teste >= 0.5).astype(int)
real_teste = y_teste.astype(int)

metricas = pd.DataFrame({
    "métrica": ["Acurácia", "Precisão", "Recall", "F1-score"],
    "valor": [
        accuracy_score(real_teste, pred_teste),
        precision_score(real_teste, pred_teste),
        recall_score(real_teste, pred_teste),
        f1_score(real_teste, pred_teste),
    ],
}).round(4)
print("Métricas no conjunto de teste (classe positiva = Cachorro):")
display(metricas)

print(classification_report(real_teste, pred_teste, target_names=["Gato", "Cachorro"], digits=4))

In [ ]:
cm = confusion_matrix(real_teste, pred_teste)
tn, fp, fn, tp = cm.ravel()

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Gato", "Cachorro"], yticklabels=["Gato", "Cachorro"])
plt.xlabel("Classe prevista")
plt.ylabel("Classe real")
plt.title("Matriz de confusão — teste")
plt.show()

print(f"Verdadeiros negativos (gato previsto como gato)        : {tn}")
print(f"Falsos positivos     (gato previsto como cachorro)     : {fp}")
print(f"Falsos negativos     (cachorro previsto como gato)     : {fn}")
print(f"Verdadeiros positivos (cachorro previsto como cachorro): {tp}")
print(f"Total de erros: {fp + fn} de {len(real_teste)} imagens")

In [ ]:
def mostrar_predicoes(indices, titulo):
    fig, axes = plt.subplots(1, len(indices), figsize=(3.4 * len(indices), 4))
    for ax, i in zip(np.atleast_1d(axes), indices):
        acertou = pred_teste[i] == real_teste[i]
        ax.imshow(X_teste[i])
        ax.set_title(f"Real: {NOMES_PT[real_teste[i]]}\nPrevista: {NOMES_PT[pred_teste[i]]}\n"
                     f"P(cachorro) = {prob_teste[i]:.2f}",
                     color="green" if acertou else "red", fontsize=10)
        ax.axis("off")
    plt.suptitle(titulo, fontsize=13)
    plt.tight_layout()
    plt.show()


rng_np = np.random.default_rng(SEED)
idx_acertos = np.where(pred_teste == real_teste)[0]
idx_erros = np.where(pred_teste != real_teste)[0]

mostrar_predicoes(rng_np.choice(idx_acertos, size=5, replace=False), "Exemplos de ACERTOS no teste")
mostrar_predicoes(rng_np.choice(idx_erros, size=min(5, len(idx_erros)), replace=False), "Exemplos de ERROS no teste")

In [ ]:
# Erros mais "confiantes": o modelo errou com probabilidade muito distante de 0,5
confianca_erro = np.abs(prob_teste[idx_erros] - 0.5)
piores = idx_erros[np.argsort(-confianca_erro)[:5]]
mostrar_predicoes(piores, "Erros com maior confiança")

exemplos = pd.DataFrame({
    "classe real": [NOMES_PT[r] for r in real_teste[:12]],
    "classe prevista": [NOMES_PT[p] for p in pred_teste[:12]],
    "P(cachorro)": prob_teste[:12].round(3),
    "acertou": ["sim" if a else "não" for a in (real_teste[:12] == pred_teste[:12])],
})
exemplos

### Leitura dos resultados

- **Acurácia:** fração de todas as imagens de teste classificadas corretamente.
- **Precisão:** das imagens que o modelo chamou de *cachorro*, quantas realmente eram.
- **Recall:** dos cachorros reais, quantos o modelo encontrou.
- **F1-score:** média harmônica entre precisão e recall.

A célula abaixo interpreta os números desta execução.

In [ ]:
acc_teste = accuracy_score(real_teste, pred_teste)
prec_teste = precision_score(real_teste, pred_teste)
rec_teste = recall_score(real_teste, pred_teste)
total_erros = fp + fn

if total_erros == 0:
    analise_erros = "O modelo não cometeu erros no conjunto de teste."
else:
    if abs(fp - fn) <= 0.15 * total_erros:
        equilibrio = ("Os dois tipos de erro aparecem em quantidade parecida, ou seja, o modelo não tem "
                      "preferência forte por uma das classes.")
    elif fp > fn:
        equilibrio = ("O erro mais frequente é o **falso positivo** (gato classificado como cachorro): o modelo "
                      "tende a responder *cachorro*, o que aumenta o recall e reduz a precisão.")
    else:
        equilibrio = ("O erro mais frequente é o **falso negativo** (cachorro classificado como gato): o modelo "
                      "tende a responder *gato*, o que aumenta a precisão e reduz o recall.")
    analise_erros = (
        f"O modelo acertou {tn + tp} das {len(real_teste)} imagens de teste (acurácia de {acc_teste:.1%}), "
        f"com precisão de {prec_teste:.1%} e recall de {rec_teste:.1%} para a classe *cachorro*. "
        f"Dos {total_erros} erros, **{fp} são falsos positivos** (gatos previstos como cachorros) e "
        f"**{fn} são falsos negativos** (cachorros previstos como gatos). {equilibrio}")

dif_val_teste = acc_teste - hist["val_accuracy"].iloc[-1]
generalizacao = (
    f"\n\nA acurácia no teste ({acc_teste:.1%}) ficou a {abs(dif_val_teste) * 100:.1f} ponto(s) percentual(is) da "
    f"acurácia de validação da última época ({hist['val_accuracy'].iloc[-1]:.1%}); como o teste não participou do "
    f"treino, esse é o desempenho que se espera em imagens novas.")

display(Markdown(analise_erros + generalizacao))

Olhando os exemplos de erro, os casos difíceis costumam ter características em comum: animal pequeno ou parcialmente
escondido na foto, fundo muito carregado (pessoas, móveis, grades), imagens escuras ou de baixa resolução, e animais
cuja pelagem ou pose lembra a outra classe. Os *erros com maior confiança* são os mais preocupantes, porque mostram
que a probabilidade da sigmoid nem sempre reflete a incerteza real do modelo.

## 8. Respostas conceituais e conclusão

### 1. O que significa o shape H × W × C de uma imagem RGB?

É o formato do tensor que representa a imagem: **H** é a altura (número de linhas de pixels), **W** é a largura (número
de colunas) e **C** é o número de canais. Em RGB, C = 3: cada pixel é descrito por três intensidades (vermelho, verde e
azul), originalmente inteiros `uint8` de 0 a 255. A imagem inspecionada na Seção 3.1 é exatamente isso: um bloco de
H × W × 3 números. Depois do pré-processamento, todas as imagens passam a ter shape `(128, 128, 3)`.

### 2. Por que as imagens foram redimensionadas para um tamanho fixo e normalizadas?

**Tamanho fixo:** as fotos originais têm dimensões diferentes, mas a rede processa lotes empilhados num único tensor
`(batch, 128, 128, 3)` e a sequência de shapes das camadas depende do tamanho de entrada. 128 × 128 preserva detalhe
suficiente para distinguir os animais e mantém o treino leve.
**Normalização:** dividir por 255 leva os pixels para [0, 1]. Entradas pequenas e na mesma escala evitam ativações e
gradientes muito grandes, tornando o treino mais rápido e estável. O mesmo tratamento foi aplicado a treino, validação e
teste (função `preparar_imagem`), para que o modelo seja avaliado em dados com a mesma representação em que aprendeu.

### 3. O que o filtro 3 × 3 aplicado manualmente destacou na imagem?

O kernel Sobel Gx `[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]` destacou as **bordas verticais**: pontos onde a intensidade
muda bruscamente da esquerda para a direita, como o contorno do animal contra o fundo, as laterais da cabeça e das
orelhas. Regiões uniformes ficaram próximas de zero e bordas horizontais praticamente não apareceram. O resultado
(126 × 126) é um mapa de características: indica **onde** o padrão procurado pelo kernel ocorre na imagem.

### 4. Como o número de filtros de uma Conv2D aparece no shape de saída?

Ele é a **última dimensão** (canais) da saída. Cada filtro produz um mapa de características; os mapas são empilhados.
Por isso `Conv2D(32)` transforma `(128, 128, 3)` em `(128, 128, 32)`, e as camadas seguintes produzem 64, 128 e 128
canais. A profundidade da entrada (3, 32, 64…) não aparece no shape de saída — ela só entra na contagem de parâmetros:
`(3·3·canais_entrada + 1) · filtros`.

### 5. Como as camadas escolhidas alteraram altura, largura e número de canais ao longo da CNN?

| Etapa | H × W | Canais |
|---|---|---|
| Entrada | 128 × 128 | 3 |
| Bloco 1 (Conv 32 + pool) | 128 × 128 → 64 × 64 | 32 |
| Bloco 2 (Conv 64 + pool) | 64 × 64 → 32 × 32 | 64 |
| Bloco 3 (Conv 128 + pool) | 32 × 32 → 16 × 16 | 128 |
| Bloco 4 (Conv 128 + pool) | 16 × 16 → 8 × 8 | 128 |

As **convoluções** usam `padding="same"` e `stride=1`, então **não alteram H e W**; elas mudam apenas o número de canais.
Quem reduz a resolução é o **MaxPooling2D 2×2**, que divide altura e largura por 2 a cada bloco (mantendo os canais),
guardando o maior valor de cada janela. Com isso o custo computacional cai, cada posição dos mapas seguintes "enxerga"
uma região maior da imagem original (campo receptivo maior) e a rede fica mais tolerante a pequenos deslocamentos do
objeto. `BatchNormalization` e `Dropout` não alteram o shape.

### 6. Como o modelo transformou os mapas de características em uma representação adequada para a camada de saída?

Com **GlobalAveragePooling2D**. Ao final do bloco 4 existem 128 mapas de 8 × 8 (tensor `(8, 8, 128)`). O GAP calcula a
**média de cada mapa**, produzindo um vetor de 128 valores — um por filtro — que indica o quanto cada padrão aprendido
está presente na imagem como um todo, independentemente da posição. Esse vetor alimenta a `Dense(64)` e, em seguida, o
neurônio sigmoid de saída. A alternativa seria o `Flatten`, que apenas enfileira os 8·8·128 = 8192 valores; ele preserva
a posição de cada ativação, mas faria a `Dense(64)` ter 524 352 pesos (contra 8 256 com GAP), aumentando bastante o
risco de overfitting.

In [ ]:
if tipo_curva == "overfitting":
    melhoria = ("**Melhoria proposta (não implementada):** combater o overfitting com *early stopping* monitorando a "
                "loss de validação (parar perto da melhor época) e data augmentation mais variado (zoom, translação, "
                "brilho), além de usar mais imagens da base original, que tem cerca de 25 mil.")
elif tipo_curva == "underfitting":
    melhoria = ("**Melhoria proposta (não implementada):** treinar por mais épocas e/ou aumentar a capacidade da rede "
                "(mais filtros ou um bloco convolucional extra), já que as curvas ainda estavam melhorando, além de "
                "usar mais imagens da base original, que tem cerca de 25 mil.")
else:
    melhoria = ("**Melhoria proposta (não implementada):** usar mais imagens da base original (cerca de 25 mil) e "
                "data augmentation mais variado (zoom, translação, brilho) para expor a rede aos casos difíceis, "
                "acompanhando com *early stopping* para não passar do ponto.")

if total_erros > 0 and abs(fp - fn) > 0.15 * total_erros:
    melhoria += (" Como os erros estão concentrados em um dos lados, também vale investigar o ajuste do limiar de "
                 "decisão (hoje fixo em 0,5) usando o conjunto de validação.")

display(Markdown(
    "### 7. As curvas de treino e validação indicam underfitting, overfitting ou comportamento adequado?\n\n"
    + diagnostico_curvas
    + "\n\n### 8. Quais tipos de erro apareceram no conjunto de teste e qual melhoria poderia ser investigada?\n\n"
    + analise_erros
    + " Visualmente, os erros se concentram em fotos com o animal pequeno ou parcialmente oculto, fundo carregado, "
      "pouca luz ou baixa resolução.\n\n"
    + melhoria
))

### Conclusão

Neste projeto acompanhamos o caminho completo de uma imagem dentro de uma CNN simples:

1. **Dado bruto → tensor:** uma foto é um bloco H × W × C de inteiros de 0 a 255; redimensionar para 128 × 128 e
   normalizar para [0, 1] deixa todas as imagens em um formato único que a rede consegue processar.
2. **Convolução:** um kernel 3 × 3 percorre a imagem e gera um mapa de características que realça um padrão local —
   vimos isso primeiro com um Sobel definido à mão e depois com os filtros aprendidos pela rede.
3. **Arquitetura:** as `Conv2D` controlam o número de canais (quantos padrões são procurados) e o `MaxPooling2D` controla
   a resolução espacial. Conseguimos prever todos os shapes e o total de parâmetros antes do `model.summary()`.
4. **Dos mapas à decisão:** o `GlobalAveragePooling2D` resume cada mapa em um valor e a camada densa com sigmoid
   converte esse resumo em uma probabilidade.
5. **Avaliação:** as curvas de treino/validação permitem diagnosticar o aprendizado, e as métricas e a matriz de
   confusão no conjunto de teste — usado apenas no final — mostram como o modelo se comporta em imagens novas e que
   tipo de erro ele comete.

Mesmo sem transfer learning, uma CNN pequena (≈ 250 mil parâmetros) treinada do zero aprende a separar gatos de
cachorros bem acima do acaso (50 %), e a análise dos erros aponta caminhos claros para evoluir o modelo.